In [ ]:
import os
import sys
import torch
import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt

from scipy.ndimage import zoom


# Make the Merlin repo importable from this notebook.
sys.path.insert(0, "/home/chest_ct/code/models/merlin")


# ============================================================
# Paths
# ============================================================

ct_path = "/home/chest_ct/code/data/data_volumes/dataset/train_fixed/train_1387_a_2.nii.gz"

gt_path = "/home/chest_ct/code/data/segmentations/segmentations/train_1387_a_2.nii.gz"


assert os.path.exists(ct_path), "CT file not found"
assert os.path.exists(gt_path), "GT mask file not found"


# ============================================================
# Load CT + Ground Truth
# ============================================================

ct = nib.load(ct_path).get_fdata()
gt = nib.load(gt_path).get_fdata()


print("CT shape:", ct.shape)
print("GT shape:", gt.shape)
print("GT labels:", np.unique(gt))


# Handle 4D masks if present
if gt.ndim == 4:
    gt = gt[0]


gt_mask = gt > 0


# ============================================================
# Load Merlin + Grad-CAM helpers
# ============================================================

from explainability.inference.load_model import load_merlin
from explainability.inference.preprocess import preprocess_ct
from explainability.inference.predict import MerlinPredictor
from explainability.gradcam.gradcam3d import GradCAM3D
from explainability.gradcam.target import ImageEmbeddingNormTarget


model, target_layer = load_merlin(image_embedding_only=True)
predictor = MerlinPredictor(model, image_embedding_only=True)

print("Merlin loaded on:", next(model.parameters()).device)


# ============================================================
# Preprocess CT
# ============================================================

image_tensor, ds_shape, affine, original_shape = preprocess_ct(ct_path)

print("Input tensor:", image_tensor.shape)
print("Downsampled shape:", ds_shape)
print("Original shape:", original_shape)


# ============================================================
# Generate Grad-CAM
# ============================================================

cam_generator = GradCAM3D(model, target_layer, predictor.forward)

cam = cam_generator(
    image_tensor,
    ImageEmbeddingNormTarget(),
    output_size=ds_shape,
)
cam_generator.release()

cam = np.asarray(cam)

print("CAM shape:", cam.shape)
print(
    "CAM range:",
    cam.min(),
    cam.max()
)


# ============================================================
# Resize CAM to CT resolution
# ============================================================

resize_factor = (
    ct.shape[0] / cam.shape[0],
    ct.shape[1] / cam.shape[1],
    ct.shape[2] / cam.shape[2]
)


cam_full = zoom(
    cam,
    resize_factor,
    order=1
)


print("Resized CAM:", cam_full.shape)


# ============================================================
# Threshold CAM
# ============================================================

threshold = 0.5

cam_mask = cam_full > threshold


print(
    "GT voxels:",
    gt_mask.sum()
)

print(
    "CAM voxels:",
    cam_mask.sum()
)


# ============================================================
# Dice
# ============================================================

intersection = np.logical_and(
    cam_mask,
    gt_mask
).sum()


dice = (
    2 * intersection /
    (
        cam_mask.sum()
        +
        gt_mask.sum()
        +
        1e-8
    )
)


# ============================================================
# IoU
# ============================================================

union = np.logical_or(
    cam_mask,
    gt_mask
).sum()


iou = intersection / (union + 1e-8)



# ============================================================
# Precision / Recall
# ============================================================

tp = np.logical_and(
    cam_mask,
    gt_mask
).sum()


fp = np.logical_and(
    cam_mask,
    ~gt_mask
).sum()


fn = np.logical_and(
    ~cam_mask,
    gt_mask
).sum()


precision = tp / (tp + fp + 1e-8)

recall = tp / (tp + fn + 1e-8)



# ============================================================
# Print Metrics
# ============================================================

print("\n===== Grad-CAM Evaluation =====")

print(f"Dice      : {dice:.4f}")
print(f"IoU       : {iou:.4f}")
print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")



# ============================================================
# Visualization
# ============================================================

slice_idx = np.argmax(
    gt_mask.sum(axis=(0,1))
)


print(
    "Best slice:",
    slice_idx
)


plt.figure(figsize=(18,6))


# CT

plt.subplot(1,3,1)

plt.imshow(
    ct[:,:,slice_idx],
    cmap="gray"
)

plt.title(
    f"CT Slice {slice_idx}"
)

plt.axis("off")



# Ground Truth

plt.subplot(1,3,2)

plt.imshow(
    ct[:,:,slice_idx],
    cmap="gray"
)


plt.imshow(
    np.ma.masked_where(
        gt_mask[:,:,slice_idx] == 0,
        gt_mask[:,:,slice_idx]
    ),
    cmap="Reds",
    alpha=0.6
)


plt.title(
    "Ground Truth Mask"
)

plt.axis("off")



# GradCAM

plt.subplot(1,3,3)

plt.imshow(
    ct[:,:,slice_idx],
    cmap="gray"
)


plt.imshow(
    cam_full[:,:,slice_idx],
    cmap="jet",
    alpha=0.5
)


plt.title(
    f"Merlin Grad-CAM\nDice={dice:.4f}, IoU={iou:.4f}"
)

plt.axis("off")


plt.tight_layout()
plt.show()

In [ ]:
import os
import sys
import torch
import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt

from scipy.ndimage import zoom


# ============================================================
# Make Merlin importable
# ============================================================

sys.path.insert(0, "/home/chest_ct/code/models/merlin")


# ============================================================
# Paths
# ============================================================

ct_path = "/home/chest_ct/code/data/data_volumes/dataset/train_fixed/train_1387_a_2.nii.gz"

gt_path = "/home/chest_ct/code/data/segmentations/segmentations/train_1387_a_2.nii.gz"


assert os.path.exists(ct_path), f"CT not found: {ct_path}"
assert os.path.exists(gt_path), f"GT not found: {gt_path}"


# ============================================================
# Load CT + Ground Truth
# ============================================================

ct_nii = nib.load(ct_path)
gt_nii = nib.load(gt_path)


ct = ct_nii.get_fdata()
gt = gt_nii.get_fdata()


if gt.ndim == 4:
    gt = gt[0]


gt_mask = gt > 0


print("CT shape :", ct.shape)
print("GT shape :", gt_mask.shape)
print("GT labels:", np.unique(gt))


# ============================================================
# Load Merlin
# ============================================================

from explainability.inference.load_model import load_merlin
from explainability.inference.preprocess import preprocess_ct
from explainability.inference.predict import MerlinPredictor
from explainability.gradcam.gradcam3d import GradCAM3D
from explainability.gradcam.target import ImageEmbeddingNormTarget


model, target_layer = load_merlin(
    image_embedding_only=True
)

predictor = MerlinPredictor(
    model,
    image_embedding_only=True
)


device = next(model.parameters()).device

print("Merlin device:", device)


# ============================================================
# Preprocess CT
# ============================================================

image_tensor, ds_shape, affine, original_shape = preprocess_ct(
    ct_path
)


image_tensor = image_tensor.to(device)


print("\nInput tensor:")
print(image_tensor.shape)

print("Preprocess output shape:")
print(ds_shape)

print("Original shape:")
print(original_shape)



# ============================================================
# Generate Grad-CAM
# ============================================================

cam_generator = GradCAM3D(
    model,
    target_layer,
    predictor.forward
)


cam = cam_generator(
    image_tensor,
    ImageEmbeddingNormTarget(),
    output_size=image_tensor.shape[2:]
)


cam_generator.release()


cam = np.asarray(cam)


print("CAM shape:", cam.shape)
print(
    "CAM range:",
    cam.min(),
    cam.max()
)



# ============================================================
# Resize CAM to CT resolution
# ============================================================

cam_full = zoom(
    cam,
    (
        ct.shape[0] / cam.shape[0],
        ct.shape[1] / cam.shape[1],
        ct.shape[2] / cam.shape[2]
    ),
    order=1
)


print("Resized CAM:", cam_full.shape)


print("\nShape verification")
print("CT       :", ct.shape)
print("GT       :", gt_mask.shape)
print("CAM full :", cam_full.shape)



# ============================================================
# Threshold CAM
# ============================================================

threshold = np.percentile(
    cam_full,
    95
)


cam_mask = cam_full >= threshold


print("CAM threshold:", threshold)

print(
    "GT voxels:",
    gt_mask.sum()
)

print(
    "CAM voxels:",
    cam_mask.sum()
)


# ============================================================
# Resize CAM to CT size
# ============================================================

resize_factor = (
    ct.shape[0] / cam.shape[0],
    ct.shape[1] / cam.shape[1],
    ct.shape[2] / cam.shape[2]
)


cam_full = zoom(
    cam,
    resize_factor,
    order=1
)


print("\nResized CAM:")
print(cam_full.shape)


print(
    "GT shape:",
    gt_mask.shape
)



# ============================================================
# Save CAM as NIfTI
# ============================================================

cam_nii = nib.Nifti1Image(
    cam_full.astype(np.float32),
    ct_nii.affine
)


nib.save(
    cam_nii,
    "train_1387_a_2_gradcam.nii.gz"
)


print(
    "Saved CAM:"
    " train_1387_a_2_gradcam.nii.gz"
)



# ============================================================
# Threshold CAM
# ============================================================

threshold = np.percentile(
    cam_full,
    95
)


cam_mask = cam_full >= threshold


print("\nThreshold:")
print(threshold)


print(
    "GT voxels:",
    gt_mask.sum()
)


print(
    "CAM voxels:",
    cam_mask.sum()
)



# ============================================================
# Dice
# ============================================================

intersection = np.logical_and(
    cam_mask,
    gt_mask
).sum()


dice = (
    2 * intersection /
    (
        cam_mask.sum()
        +
        gt_mask.sum()
        +
        1e-8
    )
)



# ============================================================
# IoU
# ============================================================

union = np.logical_or(
    cam_mask,
    gt_mask
).sum()


iou = intersection / (union + 1e-8)



# ============================================================
# Precision / Recall
# ============================================================

tp = np.logical_and(
    cam_mask,
    gt_mask
).sum()


fp = np.logical_and(
    cam_mask,
    ~gt_mask
).sum()


fn = np.logical_and(
    ~cam_mask,
    gt_mask
).sum()


precision = tp / (tp + fp + 1e-8)

recall = tp / (tp + fn + 1e-8)



# ============================================================
# Metrics
# ============================================================

print("\n==============================")
print("Grad-CAM Evaluation")
print("==============================")

print(f"Dice      : {dice:.4f}")
print(f"IoU       : {iou:.4f}")
print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")



# ============================================================
# Visualization
# ============================================================

slice_idx = np.argmax(
    gt_mask.sum(axis=(0,1))
)


print(
    "\nBest slice:",
    slice_idx
)


plt.figure(
    figsize=(18,6)
)


# CT

plt.subplot(1,3,1)

plt.imshow(
    ct[:,:,slice_idx],
    cmap="gray"
)

plt.title(
    f"CT Slice {slice_idx}"
)

plt.axis("off")



# GT

plt.subplot(1,3,2)


plt.imshow(
    ct[:,:,slice_idx],
    cmap="gray"
)


plt.imshow(
    np.ma.masked_where(
        gt_mask[:,:,slice_idx] == 0,
        gt_mask[:,:,slice_idx]
    ),
    cmap="Reds",
    alpha=0.6
)


plt.title(
    "Ground Truth"
)

plt.axis("off")



# CAM

plt.subplot(1,3,3)


plt.imshow(
    ct[:,:,slice_idx],
    cmap="gray"
)


plt.imshow(
    cam_full[:,:,slice_idx],
    cmap="jet",
    alpha=0.5
)


plt.title(
    f"Merlin Grad-CAM\nDice={dice:.4f}"
)

plt.axis("off")


plt.tight_layout()
plt.show()

In [ ]:
import os
import sys
import torch
import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt

from scipy.ndimage import zoom


# ============================================================
# Merlin import
# ============================================================

sys.path.insert(
    0,
    "/home/chest_ct/code/models/merlin"
)


# ============================================================
# Paths
# ============================================================

ct_path = "/home/chest_ct/code/data/data_volumes/dataset/train_fixed/train_1387_a_2.nii.gz"

gt_path = "/home/chest_ct/code/data/segmentations/segmentations/train_1387_a_2.nii.gz"


assert os.path.exists(ct_path)
assert os.path.exists(gt_path)



# ============================================================
# Load CT + GT
# ============================================================

ct_nii = nib.load(ct_path)
gt_nii = nib.load(gt_path)


ct = ct_nii.get_fdata()

gt = gt_nii.get_fdata()


if gt.ndim == 4:
    gt = gt[0]


gt_mask = gt > 0


print("CT shape :", ct.shape)
print("GT shape :", gt_mask.shape)
print("GT voxels:", gt_mask.sum())



# ============================================================
# Load Merlin
# ============================================================

from explainability.inference.load_model import load_merlin
from explainability.inference.preprocess import preprocess_ct
from explainability.inference.predict import MerlinPredictor
from explainability.gradcam.gradcam3d import GradCAM3D
from explainability.gradcam.target import ImageEmbeddingNormTarget



model, target_layer = load_merlin(
    image_embedding_only=True
)


predictor = MerlinPredictor(
    model,
    image_embedding_only=True
)


device = next(model.parameters()).device


print(
    "Device:",
    device
)



# ============================================================
# Preprocess
# ============================================================

image_tensor, ds_shape, affine, original_shape = preprocess_ct(
    ct_path
)


image_tensor = image_tensor.to(device)


print("\nInput:")
print(image_tensor.shape)



# ============================================================
# Grad-CAM
# ============================================================

cam_generator = GradCAM3D(
    model,
    target_layer,
    predictor.forward
)



cam = cam_generator(
    image_tensor,
    ImageEmbeddingNormTarget(),

    # FIX
    output_size=image_tensor.shape[2:]
)


cam_generator.release()


cam = np.asarray(cam)



print("\nCAM:")
print(cam.shape)



# ============================================================
# Compare GT at CAM resolution
# ============================================================

gt_small = zoom(
    gt_mask.astype(float),
    (
        cam.shape[0]/gt_mask.shape[0],
        cam.shape[1]/gt_mask.shape[1],
        cam.shape[2]/gt_mask.shape[2]
    ),
    order=0
)


gt_small = gt_small > 0.5



print("\nLow resolution comparison")

print(
    "CAM:",
    cam.shape
)

print(
    "GT small:",
    gt_small.shape
)



# ============================================================
# Threshold sweep
# ============================================================

print("\nThreshold evaluation")
print("--------------------")


for p in [80,85,90,95]:


    threshold = np.percentile(
        cam,
        p
    )


    cam_mask = cam >= threshold


    intersection = np.logical_and(
        cam_mask,
        gt_small
    ).sum()


    dice = (
        2*intersection /
        (
            cam_mask.sum()
            +
            gt_small.sum()
            +
            1e-8
        )
    )


    print(
        f"{p}%  Dice = {dice:.4f}"
    )



# ============================================================
# Best threshold
# ============================================================

threshold = np.percentile(
    cam,
    90
)


cam_mask_small = cam >= threshold



# ============================================================
# Pointing game
# ============================================================

max_point = np.unravel_index(
    np.argmax(cam),
    cam.shape
)


pointing_result = gt_small[max_point]


print("\nPointing accuracy:")
print(
    "Hit:",
    bool(pointing_result)
)



# ============================================================
# Upsample CAM to CT resolution
# ============================================================

cam_full = zoom(
    cam,
    (
        ct.shape[0]/cam.shape[0],
        ct.shape[1]/cam.shape[1],
        ct.shape[2]/cam.shape[2]
    ),
    order=1
)


print("\nFull CAM:")
print(cam_full.shape)



# ============================================================
# Save CAM
# ============================================================

cam_nii = nib.Nifti1Image(
    cam_full.astype(np.float32),
    ct_nii.affine
)


nib.save(
    cam_nii,
    "train_1387_a_2_gradcam.nii.gz"
)


print(
    "Saved CAM NIfTI"
)



# ============================================================
# Full resolution Dice
# ============================================================

cam_mask_full = cam_full >= np.percentile(
    cam_full,
    90
)



intersection = np.logical_and(
    cam_mask_full,
    gt_mask
).sum()


dice = (
    2*intersection /
    (
        cam_mask_full.sum()
        +
        gt_mask.sum()
        +
        1e-8
    )
)


print("\nFull resolution Dice:")
print(
    dice
)



# ============================================================
# Visualization
# ============================================================

slice_idx = np.argmax(
    gt_mask.sum(axis=(0,1))
)


plt.figure(
    figsize=(18,6)
)



# CT

plt.subplot(1,3,1)

plt.imshow(
    ct[:,:,slice_idx],
    cmap="gray"
)

plt.title(
    f"CT slice {slice_idx}"
)

plt.axis("off")



# GT

plt.subplot(1,3,2)

plt.imshow(
    ct[:,:,slice_idx],
    cmap="gray"
)


plt.imshow(
    np.ma.masked_where(
        gt_mask[:,:,slice_idx]==0,
        gt_mask[:,:,slice_idx]
    ),
    cmap="Reds",
    alpha=0.6
)


plt.title(
    "Ground Truth"
)

plt.axis("off")



# CAM

plt.subplot(1,3,3)

plt.imshow(
    ct[:,:,slice_idx],
    cmap="gray"
)


plt.imshow(
    cam_full[:,:,slice_idx],
    cmap="jet",
    alpha=0.5
)


plt.title(
    f"Grad-CAM\nDice={dice:.4f}"
)

plt.axis("off")


plt.tight_layout()
plt.show()

In [ ]:
import os
import sys
import torch
import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt
import torch.nn.functional as F

from scipy.ndimage import zoom, label


# ============================================================
# Merlin import
# ============================================================

sys.path.insert(
    0,
    "/home/chest_ct/code/models/merlin"
)


# ============================================================
# Paths
# ============================================================

ct_path = "/home/chest_ct/code/data/data_volumes/dataset/train_fixed/train_1387_a_2.nii.gz"

gt_path = "/home/chest_ct/code/data/segmentations/segmentations/train_1387_a_2.nii.gz"


assert os.path.exists(ct_path)
assert os.path.exists(gt_path)



# ============================================================
# Load CT + GT
# ============================================================

ct_nii = nib.load(ct_path)
gt_nii = nib.load(gt_path)

ct = ct_nii.get_fdata()
gt = gt_nii.get_fdata()

if gt.ndim == 4:
    gt = gt[0]

gt_mask = gt > 0

print("CT shape :", ct.shape)
print("GT shape :", gt_mask.shape)
print("GT voxels:", gt_mask.sum())



# ============================================================
# Load Merlin (image_embedding_only=True is fine — encode_text
# and encode_image are called directly below, bypassing the
# CLIP-style forward() that requires both image+text)
# ============================================================

from explainability.inference.load_model import load_merlin
from explainability.inference.preprocess import preprocess_ct
from explainability.gradcam.gradcam3d import GradCAM3D


model, target_layer = load_merlin(
    image_embedding_only=True
)

device = next(model.parameters()).device

print("Device:", device)



# ============================================================
# Preprocess
# ============================================================

image_tensor, ds_shape, affine, original_shape = preprocess_ct(
    ct_path
)

image_tensor = image_tensor.to(device)

print("\nInput:")
print(image_tensor.shape)



# ============================================================
# Text Similarity Target
# ============================================================

class TextSimilarityTarget:

    def __init__(self, model, text):
        self.model = model

        with torch.no_grad():
            text_embedding = model.model.encode_text([text])
            self.text_embedding = F.normalize(text_embedding, dim=-1)

    def __call__(self, output):
        image_embedding = F.normalize(output, dim=-1)
        similarity = (
            image_embedding *
            self.text_embedding.to(image_embedding.device)
        ).sum(dim=-1)
        return similarity



# ============================================================
# Image-only forward function (bypasses CLIP forward() which
# requires text; calls encode_image directly)
# ============================================================

def image_forward(x):
    return model.model.encode_image(x)



# ============================================================
# Grad-CAM
# ============================================================

cam_generator = GradCAM3D(
    model,
    target_layer,
    image_forward
)

finding = "lung nodule"

target = TextSimilarityTarget(model, finding)

cam = cam_generator(
    image_tensor,
    target,
    output_size=image_tensor.shape[2:]
)

cam_generator.release()

cam = np.asarray(cam)

print("\nCAM:")
print(cam.shape)
print("CAM range:", cam.min(), cam.max())



# ============================================================
# Compare GT at CAM resolution
# ============================================================

gt_small = zoom(
    gt_mask.astype(float),
    (
        cam.shape[0] / gt_mask.shape[0],
        cam.shape[1] / gt_mask.shape[1],
        cam.shape[2] / gt_mask.shape[2]
    ),
    order=0
)

gt_small = gt_small > 0.5

print("\nLow resolution comparison")
print("CAM:", cam.shape)
print("GT small:", gt_small.shape)



# ============================================================
# Threshold sweep
# ============================================================

print("\nThreshold evaluation")
print("--------------------")
print(f"{'pct':>4} {'dice':>8} {'cam_voxels':>10}")

for p in np.arange(50, 100, 5):

    threshold = np.percentile(cam, p)
    cam_mask = cam >= threshold

    intersection = np.logical_and(cam_mask, gt_small).sum()

    dice = (
        2 * intersection /
        (cam_mask.sum() + gt_small.sum() + 1e-8)
    )

    print(f"{p:>4} {dice:>8.4f} {cam_mask.sum():>10}")



# ============================================================
# Pointing game (single max point)
# ============================================================

max_point = np.unravel_index(np.argmax(cam), cam.shape)
pointing_result = gt_small[max_point]

print("\nPointing accuracy (single max point):")
print("Max CAM location:", max_point)
print("Hit:", bool(pointing_result))



# ============================================================
# Pointing game (top-k voxels — fairer with multiple lesions)
# ============================================================

k = 50
flat_idx = np.argsort(cam.ravel())[-k:]
top_k_coords = np.array(np.unravel_index(flat_idx, cam.shape)).T
hits = [gt_small[tuple(c)] for c in top_k_coords]

print(f"\nTop-{k} pointing hits: {sum(hits)} / {k}")



# ============================================================
# Per-lesion breakdown (GT may contain several disjoint lesions)
# ============================================================

labeled_gt, n_lesions = label(gt_small)
print(f"\nNumber of separate lesion clusters (at CAM resolution): {n_lesions}")

for i in range(1, n_lesions + 1):
    lesion_mask = labeled_gt == i
    lesion_voxels = lesion_mask.sum()
    if lesion_voxels == 0:
        continue
    cam_at_lesion = cam[lesion_mask]
    print(
        f"Lesion {i}: {lesion_voxels} voxels, "
        f"max CAM = {cam_at_lesion.max():.4f}, "
        f"mean CAM = {cam_at_lesion.mean():.4f}"
    )



# ============================================================
# Upsample CAM to CT resolution
# ============================================================

cam_full = zoom(
    cam,
    (
        ct.shape[0] / cam.shape[0],
        ct.shape[1] / cam.shape[1],
        ct.shape[2] / cam.shape[2]
    ),
    order=1
)

print("\nFull CAM:")
print(cam_full.shape)



# ============================================================
# Save CAM
# ============================================================

cam_nii = nib.Nifti1Image(
    cam_full.astype(np.float32),
    ct_nii.affine
)

nib.save(cam_nii, "train_1387_a_2_gradcam.nii.gz")

print("Saved CAM NIfTI")



# ============================================================
# Full resolution Dice
# ============================================================

cam_mask_full = cam_full >= np.percentile(cam_full, 90)

intersection = np.logical_and(cam_mask_full, gt_mask).sum()

dice = (
    2 * intersection /
    (cam_mask_full.sum() + gt_mask.sum() + 1e-8)
)

print("\nFull resolution Dice:")
print(dice)



# ============================================================
# Multi-prompt comparison (run other findings on same scan)
# ============================================================

print("\n===== Multi-prompt comparison =====")

findings = ["lung nodule", "ground glass opacity", "lung consolidation", "atelectasis"]

for f_text in findings:

    cam_generator_f = GradCAM3D(model, target_layer, image_forward)
    target_f = TextSimilarityTarget(model, f_text)

    cam_f = cam_generator_f(
        image_tensor,
        target_f,
        output_size=image_tensor.shape[2:]
    )
    cam_generator_f.release()
    cam_f = np.asarray(cam_f)

    threshold_f = np.percentile(cam_f, 90)
    cam_mask_f = cam_f >= threshold_f

    intersection_f = np.logical_and(cam_mask_f, gt_small).sum()
    dice_f = (
        2 * intersection_f /
        (cam_mask_f.sum() + gt_small.sum() + 1e-8)
    )

    max_point_f = np.unravel_index(np.argmax(cam_f), cam_f.shape)
    hit_f = bool(gt_small[max_point_f])

    print(f"{f_text:25s}  Dice@90%={dice_f:.4f}  PointingHit={hit_f}")



# ============================================================
# Visualization (using original "lung nodule" CAM)
# ============================================================

slice_idx = np.argmax(gt_mask.sum(axis=(0, 1)))

plt.figure(figsize=(18, 6))

# CT
plt.subplot(1, 3, 1)
plt.imshow(ct[:, :, slice_idx], cmap="gray")
plt.title(f"CT slice {slice_idx}")
plt.axis("off")

# GT
plt.subplot(1, 3, 2)
plt.imshow(ct[:, :, slice_idx], cmap="gray")
plt.imshow(
    np.ma.masked_where(
        gt_mask[:, :, slice_idx] == 0,
        gt_mask[:, :, slice_idx]
    ),
    cmap="Reds",
    alpha=0.6
)
plt.title("Ground Truth")
plt.axis("off")

# CAM
plt.subplot(1, 3, 3)
plt.imshow(ct[:, :, slice_idx], cmap="gray")
plt.imshow(cam_full[:, :, slice_idx], cmap="jet", alpha=0.5)
plt.title(f"Grad-CAM ({finding})\nDice={dice:.4f}")
plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
import os
import sys
import torch
import torch.nn.functional as F
import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt

from scipy.ndimage import zoom


# ============================================================
# Merlin import
# ============================================================

sys.path.insert(
    0,
    "/home/chest_ct/code/models/merlin"
)


# ============================================================
# Paths
# ============================================================

ct_path = "/home/chest_ct/code/data/data_volumes/dataset/train_fixed/train_1387_a_2.nii.gz"

gt_path = "/home/chest_ct/code/data/segmentations/segmentations/train_1387_a_2.nii.gz"


assert os.path.exists(ct_path), "CT not found"
assert os.path.exists(gt_path), "GT not found"



# ============================================================
# Load CT + Ground Truth
# ============================================================

ct_nii = nib.load(ct_path)
gt_nii = nib.load(gt_path)


ct = ct_nii.get_fdata()
gt = gt_nii.get_fdata()


if gt.ndim == 4:
    gt = gt[0]


gt_mask = gt > 0



print("CT shape :", ct.shape)
print("GT shape :", gt_mask.shape)
print("GT voxels:", gt_mask.sum())



# ============================================================
# Load Merlin
# ============================================================

from explainability.inference.load_model import load_merlin
from explainability.inference.preprocess import preprocess_ct
from explainability.gradcam.gradcam3d import GradCAM3D



model, target_layer = load_merlin(
    image_embedding_only=True
)


device = next(model.parameters()).device


print(
    "Merlin device:",
    device
)



# ============================================================
# IMPORTANT FIX
# Enable contrastive projection head
# 2048 -> 512 embedding
# ============================================================

model.model.encode_image.i3_resnet.ImageEmbedding = False



# ============================================================
# Image forward for Grad-CAM
# ============================================================

def image_forward(x):

    output = model.model.encode_image(x)

    # ImageEmbedding=False returns:
    # (image_embedding, ehr_embedding)

    image_embedding, ehr_embedding = output

    return image_embedding



# ============================================================
# Text similarity target
# ============================================================

class TextSimilarityTarget:


    def __init__(self, model, text):

        with torch.no_grad():

            text_embedding = model.model.encode_text(
                [text]
            )


            self.text_embedding = F.normalize(
                text_embedding,
                dim=-1
            )


    def __call__(self, image_embedding):

        image_embedding = F.normalize(
            image_embedding,
            dim=-1
        )


        similarity = (
            image_embedding *
            self.text_embedding.to(
                image_embedding.device
            )
        ).sum(dim=-1)


        return similarity



# ============================================================
# Preprocess CT
# ============================================================

image_tensor, ds_shape, affine, original_shape = preprocess_ct(
    ct_path
)


image_tensor = image_tensor.to(device)



print("\nInput tensor:")
print(image_tensor.shape)



# ============================================================
# Verify embeddings
# ============================================================

with torch.no_grad():

    img_test = model.model.encode_image(
        image_tensor
    )

    img_test = img_test[0]


    txt_test = model.model.encode_text(
        ["lung nodule"]
    )


print(
    "\nImage embedding:",
    img_test.shape
)

print(
    "Text embedding:",
    txt_test.shape
)



# ============================================================
# Grad-CAM
# ============================================================

cam_generator = GradCAM3D(
    model,
    target_layer,
    image_forward
)



finding = "lung nodule"


target = TextSimilarityTarget(
    model,
    finding
)



cam = cam_generator(
    image_tensor,
    target,
    output_size=image_tensor.shape[2:]
)



cam_generator.release()


cam = np.asarray(cam)



print("\nCAM:")
print(cam.shape)

print(
    "Range:",
    cam.min(),
    cam.max()
)



# ============================================================
# Resize GT to CAM resolution
# ============================================================

gt_small = zoom(
    gt_mask.astype(float),
    (
        cam.shape[0]/gt_mask.shape[0],
        cam.shape[1]/gt_mask.shape[1],
        cam.shape[2]/gt_mask.shape[2]
    ),
    order=0
)


gt_small = gt_small > 0.5



print("\nCAM resolution comparison")

print(
    "CAM:",
    cam.shape
)

print(
    "GT:",
    gt_small.shape
)



# ============================================================
# Threshold sweep
# ============================================================

print("\nThreshold evaluation")
print("--------------------")


best_dice = 0
best_threshold = 0


for p in [70,75,80,85,90,95]:


    threshold = np.percentile(
        cam,
        p
    )


    cam_mask = cam >= threshold


    intersection = np.logical_and(
        cam_mask,
        gt_small
    ).sum()


    dice = (
        2*intersection /
        (
            cam_mask.sum()
            +
            gt_small.sum()
            +
            1e-8
        )
    )


    print(
        f"{p}% Dice = {dice:.4f}"
    )


    if dice > best_dice:
        best_dice = dice
        best_threshold = threshold



print(
    "\nBest CAM threshold:",
    best_threshold
)

print(
    "Best low-res Dice:",
    best_dice
)



# ============================================================
# Pointing accuracy
# ============================================================

max_point = np.unravel_index(
    np.argmax(cam),
    cam.shape
)


hit = gt_small[max_point]


print(
    "\nPointing accuracy:",
    bool(hit)
)



# ============================================================
# Resize CAM to full CT resolution
# ============================================================

cam_full = zoom(
    cam,
    (
        ct.shape[0]/cam.shape[0],
        ct.shape[1]/cam.shape[1],
        ct.shape[2]/cam.shape[2]
    ),
    order=1
)



print(
    "\nFull CAM:",
    cam_full.shape
)



# ============================================================
# Save CAM NIfTI
# ============================================================

cam_nii = nib.Nifti1Image(
    cam_full.astype(np.float32),
    ct_nii.affine
)


nib.save(
    cam_nii,
    "train_1387_a_2_lung_nodule_gradcam.nii.gz"
)


print(
    "Saved CAM NIfTI"
)



# ============================================================
# Full resolution metrics
# ============================================================

cam_mask_full = cam_full >= best_threshold



intersection = np.logical_and(
    cam_mask_full,
    gt_mask
).sum()


union = np.logical_or(
    cam_mask_full,
    gt_mask
).sum()



dice = (
    2*intersection /
    (
        cam_mask_full.sum()
        +
        gt_mask.sum()
        +
        1e-8
    )
)


iou = intersection / (union + 1e-8)



tp = np.logical_and(
    cam_mask_full,
    gt_mask
).sum()


fp = np.logical_and(
    cam_mask_full,
    ~gt_mask
).sum()


fn = np.logical_and(
    ~cam_mask_full,
    gt_mask
).sum()



precision = tp/(tp+fp+1e-8)

recall = tp/(tp+fn+1e-8)



print("\n==============================")
print("Final Grad-CAM Evaluation")
print("==============================")

print(
    f"Dice      : {dice:.4f}"
)

print(
    f"IoU       : {iou:.4f}"
)

print(
    f"Precision : {precision:.4f}"
)

print(
    f"Recall    : {recall:.4f}"
)



# ============================================================
# Visualization
# ============================================================

slice_idx = np.argmax(
    gt_mask.sum(axis=(0,1))
)


plt.figure(
    figsize=(18,6)
)



# CT

plt.subplot(1,3,1)

plt.imshow(
    ct[:,:,slice_idx],
    cmap="gray"
)

plt.title(
    f"CT Slice {slice_idx}"
)

plt.axis("off")



# Ground truth

plt.subplot(1,3,2)

plt.imshow(
    ct[:,:,slice_idx],
    cmap="gray"
)


plt.imshow(
    np.ma.masked_where(
        gt_mask[:,:,slice_idx]==0,
        gt_mask[:,:,slice_idx]
    ),
    cmap="Reds",
    alpha=0.6
)


plt.title(
    "Ground Truth"
)

plt.axis("off")



# GradCAM

plt.subplot(1,3,3)

plt.imshow(
    ct[:,:,slice_idx],
    cmap="gray"
)


plt.imshow(
    cam_full[:,:,slice_idx],
    cmap="jet",
    alpha=0.5
)


plt.title(
    f"Merlin Grad-CAM\nDice={dice:.4f}"
)

plt.axis("off")



plt.tight_layout()

plt.show()

In [ ]:
import os
import sys
import torch
import torch.nn.functional as F
import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt

from scipy.ndimage import zoom


# ============================================================
# Merlin import
# ============================================================

sys.path.insert(
    0,
    "/home/chest_ct/code/models/merlin"
)


# ============================================================
# Paths
# ============================================================

ct_path = "/home/chest_ct/code/data/data_volumes/dataset/train_fixed/train_1387_a_2.nii.gz"

gt_path = "/home/chest_ct/code/data/segmentations/segmentations/train_1387_a_2.nii.gz"


assert os.path.exists(ct_path), "CT not found"
assert os.path.exists(gt_path), "GT not found"



# ============================================================
# Load CT + Ground Truth
# ============================================================

ct_nii = nib.load(ct_path)
gt_nii = nib.load(gt_path)


ct = ct_nii.get_fdata()
gt = gt_nii.get_fdata()


if gt.ndim == 4:
    gt = gt[0]


gt_mask = gt > 0



print("CT shape :", ct.shape)
print("GT shape :", gt_mask.shape)
print("GT voxels:", gt_mask.sum())



# ============================================================
# Load Merlin
# ============================================================

from explainability.inference.load_model import load_merlin
from explainability.inference.preprocess import preprocess_ct
from explainability.gradcam.gradcam3d import GradCAM3D



model, target_layer = load_merlin(
    image_embedding_only=True
)


device = next(model.parameters()).device


print(
    "Merlin device:",
    device
)



# ============================================================
# IMPORTANT FIX
# Enable contrastive projection head
# 2048 -> 512 embedding
# ============================================================

model.model.encode_image.i3_resnet.ImageEmbedding = False



# ============================================================
# Image forward for Grad-CAM
# ============================================================

def image_forward(x):

    output = model.model.encode_image(x)

    # ImageEmbedding=False returns:
    # (image_embedding, ehr_embedding)

    image_embedding, ehr_embedding = output

    return image_embedding



# ============================================================
# Text similarity target
# ============================================================

class TextSimilarityTarget:


    def __init__(self, model, text):

        with torch.no_grad():

            text_embedding = model.model.encode_text(
                [text]
            )


            self.text_embedding = F.normalize(
                text_embedding,
                dim=-1
            )


    def __call__(self, image_embedding):

        image_embedding = F.normalize(
            image_embedding,
            dim=-1
        )


        similarity = (
            image_embedding *
            self.text_embedding.to(
                image_embedding.device
            )
        ).sum(dim=-1)


        return similarity



# ============================================================
# Preprocess CT
# ============================================================

image_tensor, ds_shape, affine, original_shape = preprocess_ct(
    ct_path
)


image_tensor = image_tensor.to(device)


# ============================================================
# CRITICAL FIX
# Without this, torch.utils.checkpoint used inside I3ResNet's
# layer1-layer4 cannot properly backpropagate, gradients come
# back as None, and Grad-CAM silently produces a meaningless map
# ============================================================

image_tensor.requires_grad_(True)



print("\nInput tensor:")
print(image_tensor.shape)
print("requires_grad:", image_tensor.requires_grad)



# ============================================================
# Verify embeddings (sanity check only — no_grad here is fine,
# this is separate from the actual Grad-CAM forward pass below)
# ============================================================

with torch.no_grad():

    img_test = model.model.encode_image(
        image_tensor
    )

    img_test = img_test[0]


    txt_test = model.model.encode_text(
        ["lung nodule"]
    )


print(
    "\nImage embedding:",
    img_test.shape
)

print(
    "Text embedding:",
    txt_test.shape
)



# ============================================================
# Grad-CAM
# ============================================================

cam_generator = GradCAM3D(
    model,
    target_layer,
    image_forward
)



finding = "lung nodule"


target = TextSimilarityTarget(
    model,
    finding
)



cam = cam_generator(
    image_tensor,
    target,
    output_size=image_tensor.shape[2:]
)



cam_generator.release()


cam = np.asarray(cam)



print("\nCAM:")
print(cam.shape)

print(
    "Range:",
    cam.min(),
    cam.max()
)

print(
    "Std:",
    cam.std()
)

# A near-zero std here would still indicate a degenerate/flat
# CAM even after fixing requires_grad — worth checking before
# trusting the Dice numbers below.



# ============================================================
# Resize GT to CAM resolution
# ============================================================

gt_small = zoom(
    gt_mask.astype(float),
    (
        cam.shape[0]/gt_mask.shape[0],
        cam.shape[1]/gt_mask.shape[1],
        cam.shape[2]/gt_mask.shape[2]
    ),
    order=0
)


gt_small = gt_small > 0.5



print("\nCAM resolution comparison")

print(
    "CAM:",
    cam.shape
)

print(
    "GT:",
    gt_small.shape
)



# ============================================================
# Threshold sweep
# ============================================================

print("\nThreshold evaluation")
print("--------------------")


best_dice = 0
best_threshold = 0


for p in [70,75,80,85,90,95]:


    threshold = np.percentile(
        cam,
        p
    )


    cam_mask = cam >= threshold


    intersection = np.logical_and(
        cam_mask,
        gt_small
    ).sum()


    dice = (
        2*intersection /
        (
            cam_mask.sum()
            +
            gt_small.sum()
            +
            1e-8
        )
    )


    print(
        f"{p}% Dice = {dice:.4f}"
    )


    if dice > best_dice:
        best_dice = dice
        best_threshold = threshold



print(
    "\nBest CAM threshold:",
    best_threshold
)

print(
    "Best low-res Dice:",
    best_dice
)



# ============================================================
# Pointing accuracy
# ============================================================

max_point = np.unravel_index(
    np.argmax(cam),
    cam.shape
)


hit = gt_small[max_point]


print(
    "\nPointing accuracy:",
    bool(hit)
)



# ============================================================
# Resize CAM to full CT resolution
# ============================================================

cam_full = zoom(
    cam,
    (
        ct.shape[0]/cam.shape[0],
        ct.shape[1]/cam.shape[1],
        ct.shape[2]/cam.shape[2]
    ),
    order=1
)



print(
    "\nFull CAM:",
    cam_full.shape
)



# ============================================================
# Save CAM NIfTI
# ============================================================

cam_nii = nib.Nifti1Image(
    cam_full.astype(np.float32),
    ct_nii.affine
)


nib.save(
    cam_nii,
    "train_1387_a_2_lung_nodule_gradcam.nii.gz"
)


print(
    "Saved CAM NIfTI"
)



# ============================================================
# Full resolution metrics
# ============================================================

cam_mask_full = cam_full >= best_threshold



intersection = np.logical_and(
    cam_mask_full,
    gt_mask
).sum()


union = np.logical_or(
    cam_mask_full,
    gt_mask
).sum()



dice = (
    2*intersection /
    (
        cam_mask_full.sum()
        +
        gt_mask.sum()
        +
        1e-8
    )
)


iou = intersection / (union + 1e-8)



tp = np.logical_and(
    cam_mask_full,
    gt_mask
).sum()


fp = np.logical_and(
    cam_mask_full,
    ~gt_mask
).sum()


fn = np.logical_and(
    ~cam_mask_full,
    gt_mask
).sum()



precision = tp/(tp+fp+1e-8)

recall = tp/(tp+fn+1e-8)



print("\n==============================")
print("Final Grad-CAM Evaluation")
print("==============================")

print(
    f"Dice      : {dice:.4f}"
)

print(
    f"IoU       : {iou:.4f}"
)

print(
    f"Precision : {precision:.4f}"
)

print(
    f"Recall    : {recall:.4f}"
)



# ============================================================
# Visualization
# ============================================================

slice_idx = np.argmax(
    gt_mask.sum(axis=(0,1))
)


plt.figure(
    figsize=(18,6)
)



# CT

plt.subplot(1,3,1)

plt.imshow(
    ct[:,:,slice_idx],
    cmap="gray"
)

plt.title(
    f"CT Slice {slice_idx}"
)

plt.axis("off")



# Ground truth

plt.subplot(1,3,2)

plt.imshow(
    ct[:,:,slice_idx],
    cmap="gray"
)


plt.imshow(
    np.ma.masked_where(
        gt_mask[:,:,slice_idx]==0,
        gt_mask[:,:,slice_idx]
    ),
    cmap="Reds",
    alpha=0.6
)


plt.title(
    "Ground Truth"
)

plt.axis("off")



# GradCAM

plt.subplot(1,3,3)

plt.imshow(
    ct[:,:,slice_idx],
    cmap="gray"
)


plt.imshow(
    cam_full[:,:,slice_idx],
    cmap="jet",
    alpha=0.5
)


plt.title(
    f"Merlin Grad-CAM\nDice={dice:.4f}"
)

plt.axis("off")



plt.tight_layout()

plt.show()

In [ ]:
positive_slices = np.where(
    gt_mask.sum(axis=(0,1)) > 0
)

print(
    "GT positive slices:",
    positive_slices[0]
)

In [ ]:
max_point = np.unravel_index(
    np.argmax(cam),
    cam.shape
)

print(
    "Maximum CAM location:",
    max_point
)

print(
    "GT at maximum point:",
    gt_small[max_point]
)

In [ ]:
thresholds = np.arange(50,100,5)

for p in thresholds:

    threshold = np.percentile(cam,p)

    cam_mask = cam >= threshold

    intersection = np.logical_and(
        cam_mask,
        gt_small
    ).sum()

    dice = (
        2*intersection /
        (
            cam_mask.sum()
            +
            gt_small.sum()
            +
            1e-8
        )
    )

    print(
        p,
        dice,
        cam_mask.sum()
    )

In [ ]:
k = 50
flat_idx = np.argsort(cam.ravel())[-k:]
top_k_coords = np.array(np.unravel_index(flat_idx, cam.shape)).T

hits = [gt_small[tuple(c)] for c in top_k_coords]
print(f"Top-{k} pointing hits: {sum(hits)} / {k}")

In [ ]:
thresholds = np.arange(50, 100, 5)

print(f"{'pct':>4} {'dice':>8} {'cam_voxels':>10}")
for p in thresholds:
    threshold = np.percentile(cam, p)
    cam_mask = cam >= threshold

    intersection = np.logical_and(cam_mask, gt_small).sum()
    dice = (
        2 * intersection /
        (cam_mask.sum() + gt_small.sum() + 1e-8)
    )

    print(f"{p:>4} {dice:>8.4f} {cam_mask.sum():>10}")

In [ ]:
from scipy.ndimage import label

labeled_gt, n_lesions = label(gt_small)
print(f"Number of separate lesion clusters: {n_lesions}")

for i in range(1, n_lesions + 1):
    lesion_mask = labeled_gt == i
    lesion_voxels = lesion_mask.sum()
    cam_at_lesion = cam[lesion_mask]
    print(f"Lesion {i}: {lesion_voxels} voxels, "
          f"max CAM value inside = {cam_at_lesion.max():.4f}, "
          f"mean CAM value inside = {cam_at_lesion.mean():.4f}")

In [ ]:
findings = ["lung nodule", "ground glass opacity", "lung consolidation", "atelectasis"]

for finding in findings:
    target = TextSimilarityTarget(model, finding)
    cam_f = cam_generator(image_tensor, target, output_size=image_tensor.shape[2:])
    cam_generator.release()
    cam_f = np.asarray(cam_f)

    threshold = np.percentile(cam_f, 90)
    cam_mask = cam_f >= threshold
    intersection = np.logical_and(cam_mask, gt_small).sum()
    dice = 2*intersection / (cam_mask.sum() + gt_small.sum() + 1e-8)

    max_point = np.unravel_index(np.argmax(cam_f), cam_f.shape)
    hit = bool(gt_small[max_point])

    print(f"{finding:25s}  Dice@90%={dice:.4f}  PointingHit={hit}")

In [ ]:
import os
import sys
import torch
import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt

from scipy.ndimage import zoom


# ============================================================
# Merlin import
# ============================================================

sys.path.insert(
    0,
    "/home/chest_ct/code/models/merlin"
)


# ============================================================
# Paths
# ============================================================

ct_path = "/home/chest_ct/code/data/data_volumes/dataset/train_fixed/train_1387_a_2.nii.gz"

gt_path = "/home/chest_ct/code/data/segmentations/segmentations/train_1387_a_2.nii.gz"


assert os.path.exists(ct_path)
assert os.path.exists(gt_path)



# ============================================================
# Load CT + GT
# ============================================================

ct_nii = nib.load(ct_path)
gt_nii = nib.load(gt_path)


ct = ct_nii.get_fdata()

gt = gt_nii.get_fdata()


if gt.ndim == 4:
    gt = gt[0]


gt_mask = gt > 0


print("CT shape :", ct.shape)
print("GT shape :", gt_mask.shape)
print("GT voxels:", gt_mask.sum())



# ============================================================
# Load Merlin
# ============================================================

from explainability.inference.load_model import load_merlin
from explainability.inference.preprocess import preprocess_ct
from explainability.inference.predict import MerlinPredictor
from explainability.gradcam.gradcam3d import GradCAM3D

import torch.nn.functional as F



model, target_layer = load_merlin(
    image_embedding_only=False
)


predictor = MerlinPredictor(
    model,
    image_embedding_only=True
)




device = next(model.parameters()).device

# ============================================================
# Text Similarity Target
# ============================================================

class TextSimilarityTarget:

    def __init__(self, model, text):

        self.model = model

        with torch.no_grad():

            text_embedding = model.model.encode_text(
                [text]
            )

            self.text_embedding = F.normalize(
                text_embedding,
                dim=-1
            )


    def __call__(self, output):

        image_embedding = F.normalize(
            output,
            dim=-1
        )


        similarity = (
            image_embedding *
            self.text_embedding.to(
                image_embedding.device
            )
        ).sum(dim=-1)


        return similarity
print(
    "Device:",
    device
)



# ============================================================
# Preprocess
# ============================================================

image_tensor, ds_shape, affine, original_shape = preprocess_ct(
    ct_path
)


image_tensor = image_tensor.to(device)


print("\nInput:")
print(image_tensor.shape)



# ============================================================
# Grad-CAM
# ============================================================

cam_generator = GradCAM3D(
    model,
    target_layer,
    predictor.forward
)


finding = "lung nodule"


target = TextSimilarityTarget(
    model,
    finding
)


cam = cam_generator(
    image_tensor,
    target,
    output_size=image_tensor.shape[2:]
)

cam_generator.release()


cam = np.asarray(cam)



print("\nCAM:")
print(cam.shape)



# ============================================================
# Compare GT at CAM resolution
# ============================================================

gt_small = zoom(
    gt_mask.astype(float),
    (
        cam.shape[0]/gt_mask.shape[0],
        cam.shape[1]/gt_mask.shape[1],
        cam.shape[2]/gt_mask.shape[2]
    ),
    order=0
)


gt_small = gt_small > 0.5



print("\nLow resolution comparison")

print(
    "CAM:",
    cam.shape
)

print(
    "GT small:",
    gt_small.shape
)



# ============================================================
# Threshold sweep
# ============================================================

print("\nThreshold evaluation")
print("--------------------")


for p in [80,85,90,95]:


    threshold = np.percentile(
        cam,
        p
    )


    cam_mask = cam >= threshold


    intersection = np.logical_and(
        cam_mask,
        gt_small
    ).sum()


    dice = (
        2*intersection /
        (
            cam_mask.sum()
            +
            gt_small.sum()
            +
            1e-8
        )
    )


    print(
        f"{p}%  Dice = {dice:.4f}"
    )



# ============================================================
# Best threshold
# ============================================================

threshold = np.percentile(
    cam,
    90
)


cam_mask_small = cam >= threshold



# ============================================================
# Pointing game
# ============================================================

max_point = np.unravel_index(
    np.argmax(cam),
    cam.shape
)


pointing_result = gt_small[max_point]


print("\nPointing accuracy:")
print(
    "Hit:",
    bool(pointing_result)
)



# ============================================================
# Upsample CAM to CT resolution
# ============================================================

cam_full = zoom(
    cam,
    (
        ct.shape[0]/cam.shape[0],
        ct.shape[1]/cam.shape[1],
        ct.shape[2]/cam.shape[2]
    ),
    order=1
)


print("\nFull CAM:")
print(cam_full.shape)



# ============================================================
# Save CAM
# ============================================================

cam_nii = nib.Nifti1Image(
    cam_full.astype(np.float32),
    ct_nii.affine
)


nib.save(
    cam_nii,
    "train_1387_a_2_gradcam.nii.gz"
)


print(
    "Saved CAM NIfTI"
)



# ============================================================
# Full resolution Dice
# ============================================================

cam_mask_full = cam_full >= np.percentile(
    cam_full,
    90
)



intersection = np.logical_and(
    cam_mask_full,
    gt_mask
).sum()


dice = (
    2*intersection /
    (
        cam_mask_full.sum()
        +
        gt_mask.sum()
        +
        1e-8
    )
)


print("\nFull resolution Dice:")
print(
    dice
)



# ============================================================
# Visualization
# ============================================================

slice_idx = np.argmax(
    gt_mask.sum(axis=(0,1))
)


plt.figure(
    figsize=(18,6)
)



# CT

plt.subplot(1,3,1)

plt.imshow(
    ct[:,:,slice_idx],
    cmap="gray"
)

plt.title(
    f"CT slice {slice_idx}"
)

plt.axis("off")



# GT

plt.subplot(1,3,2)

plt.imshow(
    ct[:,:,slice_idx],
    cmap="gray"
)


plt.imshow(
    np.ma.masked_where(
        gt_mask[:,:,slice_idx]==0,
        gt_mask[:,:,slice_idx]
    ),
    cmap="Reds",
    alpha=0.6
)


plt.title(
    "Ground Truth"
)

plt.axis("off")



# CAM

plt.subplot(1,3,3)

plt.imshow(
    ct[:,:,slice_idx],
    cmap="gray"
)


plt.imshow(
    cam_full[:,:,slice_idx],
    cmap="jet",
    alpha=0.5
)


plt.title(
    f"Grad-CAM\nDice={dice:.4f}"
)

plt.axis("off")


plt.tight_layout()
plt.show()

In [ ]:
import os
import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt

from scipy.stats import pearsonr
from sklearn.metrics.pairwise import cosine_similarity


# ============================================================
# Paths
# ============================================================

official_cam_path = "/home/chest_ct/code/models/merlin/gradcam_outputs/official_gradcam.nii.gz"

my_cam_path = "/home/chest_ct/code/models/merlin/my_gradcam.nii.gz"

ct_path = "/home/chest_ct/code/data/data_volumes/dataset/train_fixed/train_1387_a_2.nii.gz"

gt_path = "/home/chest_ct/code/data/segmentations/segmentations/train_1387_a_2.nii.gz"



assert os.path.exists(official_cam_path), "Official CAM not found"
assert os.path.exists(my_cam_path), "Your CAM not found"
assert os.path.exists(ct_path), "CT not found"
assert os.path.exists(gt_path), "GT not found"



# ============================================================
# Load files
# ============================================================

official_cam = nib.load(
    official_cam_path
).get_fdata()


my_cam = nib.load(
    my_cam_path
).get_fdata()


ct = nib.load(
    ct_path
).get_fdata()


gt = nib.load(
    gt_path
).get_fdata()


if gt.ndim == 4:
    gt = gt[0]


gt_mask = gt > 0



print("Official CAM:", official_cam.shape)
print("My CAM      :", my_cam.shape)
print("CT          :", ct.shape)
print("GT          :", gt_mask.shape)



# ============================================================
# Shape check
# ============================================================

assert official_cam.shape == my_cam.shape, \
    "CAM shapes are different"


assert official_cam.shape == ct.shape, \
    "CAM and CT shapes are different"



# ============================================================
# Normalize CAMs
# ============================================================

def normalize(cam):

    cam = cam - cam.min()

    cam = cam / (
        cam.max() + 1e-8
    )

    return cam



official_cam = normalize(
    official_cam
)


my_cam = normalize(
    my_cam
)



# ============================================================
# Pearson correlation
# ============================================================

pearson, p = pearsonr(
    official_cam.flatten(),
    my_cam.flatten()
)


print("\n============================")
print("CAM Similarity")
print("============================")


print(
    f"Pearson correlation : {pearson:.6f}"
)



# ============================================================
# Cosine similarity
# ============================================================

cos = cosine_similarity(
    official_cam.reshape(1,-1),
    my_cam.reshape(1,-1)
)[0][0]


print(
    f"Cosine similarity   : {cos:.6f}"
)



# ============================================================
# Mean Absolute Error
# ============================================================

mae = np.mean(
    np.abs(
        official_cam -
        my_cam
    )
)


print(
    f"MAE                 : {mae:.6f}"
)



# ============================================================
# Threshold comparison
# ============================================================

threshold = 90


official_mask = (
    official_cam >=
    np.percentile(
        official_cam,
        threshold
    )
)


my_mask = (
    my_cam >=
    np.percentile(
        my_cam,
        threshold
    )
)



def dice_score(a,b):

    intersection = np.logical_and(
        a,b
    ).sum()


    return (
        2*intersection /
        (
            a.sum()
            +
            b.sum()
            +
            1e-8
        )
    )



print("\nCAM binary overlap")
print("------------------")


print(
    "Dice:",
    dice_score(
        official_mask,
        my_mask
    )
)



# ============================================================
# Find lesion slice
# ============================================================

slice_idx = np.argmax(
    gt_mask.sum(axis=(0,1))
)


print(
    "\nVisualization slice:",
    slice_idx
)



# ============================================================
# Visualization
# ============================================================

plt.figure(
    figsize=(20,8)
)



# CT

plt.subplot(2,3,1)

plt.imshow(
    ct[:,:,slice_idx],
    cmap="gray"
)

plt.title(
    "CT"
)

plt.axis("off")



# GT

plt.subplot(2,3,2)

plt.imshow(
    ct[:,:,slice_idx],
    cmap="gray"
)


plt.imshow(
    np.ma.masked_where(
        gt_mask[:,:,slice_idx]==0,
        gt_mask[:,:,slice_idx]
    ),
    cmap="Reds",
    alpha=0.6
)


plt.title(
    "Ground Truth"
)

plt.axis("off")



# Official CAM

plt.subplot(2,3,3)

plt.imshow(
    ct[:,:,slice_idx],
    cmap="gray"
)


plt.imshow(
    official_cam[:,:,slice_idx],
    cmap="jet",
    alpha=0.5
)


plt.title(
    "Official Merlin Grad-CAM"
)

plt.axis("off")



# Your CAM

plt.subplot(2,3,4)

plt.imshow(
    ct[:,:,slice_idx],
    cmap="gray"
)


plt.imshow(
    my_cam[:,:,slice_idx],
    cmap="jet",
    alpha=0.5
)


plt.title(
    "Your Grad-CAM"
)

plt.axis("off")



# Difference

plt.subplot(2,3,5)

plt.imshow(
    np.abs(
        official_cam[:,:,slice_idx]
        -
        my_cam[:,:,slice_idx]
    ),
    cmap="hot"
)


plt.title(
    "Absolute Difference"
)

plt.axis("off")



# Scatter

plt.subplot(2,3,6)


plt.scatter(
    official_cam.flatten()[::100],
    my_cam.flatten()[::100],
    s=1
)

 
plt.xlabel(
    "Official CAM"
)

plt.ylabel(
    "Your CAM"
)

plt.title(
    "CAM Correlation"
)



plt.tight_layout()

plt.show()